<a href="https://colab.research.google.com/github/MarcinPlodzien/deep-learning-for-scientific-discovery/blob/main/notebooks/lecture-07/exercise-sp500.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Forecasting the S&P 500 index

*Deep Learning for Scientific Discovery, Jagiellonian University. Author: Marcin Płodzień.*

The Lorenz-63 flow, the Mackey–Glass delay equation and the Santa Fe laser all obey equations of motion, and their future is fixed by the present state. Our models learned an approximation to that law from the record alone, and on the Mackey–Glass signals the LSTM reached a total capacity of 49.9 out of 50. Many signals measured in science are of a different kind. The position of a pollen grain in water, the number of photons counted by a detector in one millisecond, the noise voltage of a resistor or the daily rainfall at one station are driven by a very large number of influences that nobody records. Each value is then best described as a random draw, and the signal as a whole as a stochastic process. A forecaster applied to such a signal must identify the part of the future fixed by the past and report zero skill when that part is empty.

Financial prices are the oldest and best studied signal of this kind. In 1900 Louis Bachelier modelled the prices of the Paris stock exchange as a random walk, a sum of independent random steps, in his doctoral thesis [L. Bachelier, Ann. Sci. Éc. Norm. Supér. **17**, 21 (1900), [doi:10.24033/asens.476](https://doi.org/10.24033/asens.476)], five years before Einstein used the same mathematics for Brownian motion [A. Einstein, Ann. Phys. **17**, 549 (1905), [doi:10.1002/andp.19053220806](https://doi.org/10.1002/andp.19053220806)]. The financial value of a forecast depends on predicting price changes. A trader who knew tomorrow's change of a price would know whether to buy or to sell today, and a model that is right slightly more often than chance can be worth a great deal of money. Many people have looked for such patterns, and the efficient-market hypothesis states the consequence [E. F. Fama, J. Finance **25**, 383 (1970), [doi:10.1111/j.1540-6261.1970.tb00518.x](https://doi.org/10.1111/j.1540-6261.1970.tb00518.x)]. A predictable pattern in the price is used by traders as soon as it is found, their buying and selling moves the price, and the pattern disappears. The price then reflects the information available today, and its next change is close to unpredictable from the past prices.

We test the forecasting methods on the daily closing values of the S&P 500 index over ten years, using the same models, chronological split and scores as for Mackey–Glass and the Santa Fe laser. The models may identify structure in past values or report an absence of structure. They can also report high skill on a signal whose changes are unpredictable. The price-level forecasts produce this misleading outcome, which requires careful interpretation of the score.

We compare forecasts of four signals, beginning with a synthetic stochastic process whose capacity is known from a formula. This provides a check on the forecasting procedure. Forecasts of the price level of the index reach a high capacity that carries little information about the future changes of the index. The daily logarithmic returns, the relative price changes that a trader actually needs to predict, are forecast with all models of the lecture, from the autoregressive (AR), moving-average (MA) and combined (ARMA) models through the nonlinear vector autoregression (NVAR) to a multilayer perceptron (MLP), an Elman recurrent neural network (RNN) and a long short-term memory network (LSTM), and compared with the same returns shuffled in time. The size of the daily price change is partly predictable even when its sign is unpredictable. The central England temperature and sunspot analyses apply the same methods to two measured records and include implementations of the calculations.

For every signal and model we plot the forecasts against the true values at the horizons τ = 1, 5 and 20 trading days, plot the capacity C(τ) against τ and collect the total capacities in one table. The models are implemented by the reader, following the lecture notebooks on linear models and NVAR, on the MLP forecaster, on the recurrent network and on the LSTM. The data loading, the scores and the plotting functions are given.

> **Google Colab execution**
> 1. Go to <https://colab.research.google.com> → *File → Upload notebook*. The setting `JAX_PLATFORMS=cpu` selects CPU execution, which takes a few minutes.
> 2. `jax`, `flax`, `optax`, `statsmodels`, `scikit-learn`, `pandas` and `matplotlib` are pre-installed in Colab.
> 3. The S&P 500 data are downloaded from FRED during execution, so an internet connection is needed.
> 4. Run the cells *in order*. Later cells use variables defined earlier.

In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"     # run JAX on the CPU (set before importing jax)
import time, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy.stats
import jax, jax.numpy as jnp
import optax
import flax
from flax import nnx
import sklearn
from sklearn.neural_network import MLPRegressor
import statsmodels
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf
warnings.filterwarnings("ignore", module="statsmodels")
warnings.filterwarnings("ignore", category=sklearn.exceptions.ConvergenceWarning)

plt.rcParams.update({"font.size": 15, "axes.titlesize": 16, "axes.labelsize": 15, "xtick.labelsize": 13,
                     "ytick.labelsize": 13, "legend.fontsize": 13, "figure.titlesize": 17,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})
UJBlue, UJNavy, DSOrange, UJGray = "#005CA7", "#002B5C", "#D35A05", "#5A6270"
os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)
t_start = time.time()
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__,
      "| statsmodels", statsmodels.__version__, "| scikit-learn", sklearn.__version__)

<hr/>
## 1. The S&P 500 index

The S&P 500 is a stock-market index computed by S&P Dow Jones Indices. A share is a unit of ownership of a company, and its price is set every second by the trades on a stock exchange. The index combines the share prices of 500 leading companies of the United States, listed on the New York Stock Exchange or on NASDAQ, into one number, in which every company enters with a weight given by its total market value. The 500 companies cover about 75% of the value of all shares traded in the United States, so the index is the standard gauge of the large companies of the American economy. It is a price index. Dividends, the parts of the profit that the companies pay to their shareholders, are left out.

One sample of our record is one number, the value of the index at the close of trading on one day, in index points. The market closes at 4 pm New York time. There is no trading on weekends and on public holidays, so the record has one value per trading day, about 252 per year, and consecutive samples can be one, three or four calendar days apart. We use the ten years from 28 October 2016 to 27 October 2026. The record has no labels, and the task is to forecast later values of the record from earlier ones.

The data come from FRED, the economic database of the Federal Reserve Bank of St. Louis, which distributes the series under the code SP500.

> S&P Dow Jones Indices LLC, *S&P 500* [SP500], retrieved from FRED, Federal Reserve Bank of St. Louis, <https://fred.stlouisfed.org/series/SP500>.

The series is copyrighted by S&P Dow Jones Indices LLC, and its reproduction without their permission is prohibited. For this reason the data are not distributed with the course. The code downloads them from FRED for your own use each time it runs, and FRED makes ten years of daily history available. The download requests a fixed date range, so that everyone works with the same record. Days without a value, the market holidays, are marked by an empty field and dropped.

In [ ]:
FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=SP500&cosd=2016-10-28&coed=2026-10-27"
SP_FILE = "data/sp500_fred.csv"
if not os.path.exists(SP_FILE):
    urllib.request.urlretrieve(FRED_URL, SP_FILE)
raw_sp = pd.read_csv(SP_FILE, na_values=["", "."], parse_dates=["observation_date"])
sp = raw_sp.dropna().reset_index(drop=True)
dates, price = sp["observation_date"].to_numpy(), sp["SP500"].to_numpy(float)
print(f"{len(raw_sp)} rows, {len(raw_sp) - len(sp)} without a value, {len(price)} trading days")
print(f"first day {sp.observation_date.iloc[0].date()}: {price[0]:.2f} points, "
      f"last day {sp.observation_date.iloc[-1].date()}: {price[-1]:.2f} points")
gaps = np.diff(sp["observation_date"]).astype("timedelta64[D]").astype(int)
print("calendar days between consecutive trading days:", dict(zip(*np.unique(gaps, return_counts=True))))
print(sp.head(5).to_string(index=False))

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(12, 7), layout="constrained")
axs[0].plot(dates, price, color=UJBlue, lw=1.2)
axs[0].set_ylabel("S&P 500 [points]")
axs[0].set_title("daily closing value, ten years")
sel = slice(len(price) - 60, len(price))
axs[1].plot(dates[sel], price[sel], "o-", color=UJBlue, ms=4, lw=1)
axs[1].set_ylabel("S&P 500 [points]")
axs[1].set_title("the last 60 trading days")
fig.autofmt_xdate()
plt.show()

Over the whole record, the index rises from about two thousand to about seven and a half thousand points with long periods of small changes and short periods of steep falls. The largest fall, in February and March 2020, came with the outbreak of the COVID-19 pandemic. The individual values over the last 60 trading days form a sequence of small irregular steps up and down.

<hr/>
## 2. Stochastic processes

A stochastic process is a sequence of random variables $s_1,s_2,s_3,\dots$, one for every time step. The measured record is one realisation of it, one particular sequence of numbers out of all sequences that the process could have produced. A deterministic signal is the special case in which the whole sequence is fixed by the initial state. For a stochastic process the past fixes at most a probability distribution of the next value, and the best possible forecast is the mean of this distribution given everything known so far,
$$\hat s_{k+\tau}=\mathbb E\big[s_{k+\tau}\,\big|\,s_k,s_{k-1},\dots\big] . \tag{1}$$
The conditional mean $\mathbb E[\,\cdot\,|\,\cdot\,]$ is the average over all realisations that share the observed past. It minimises the mean squared error of the forecast, and no model, however large, can do better on average. The remaining part, $s_{k+\tau}-\hat s_{k+\tau}$, is the part of the future that the past does not fix. For a deterministic chaotic signal this remainder starts at zero and grows with τ as small errors are amplified. For a stochastic process it is already present at τ = 1.

The simplest stochastic process with memory is the autoregressive process of order one, AR(1),
$$s_k=\phi\,s_{k-1}+\xi_k , \tag{2}$$
in which each value is a fraction φ of the previous one plus a new random kick $\xi_k$. The kicks are independent Gaussian numbers with mean zero and variance $\sigma_\xi^2$, a sequence called white noise. For $|\phi|<1$ the process forgets its past at a constant rate. Its best forecast follows from equation (2) applied τ times, with the unknown future kicks replaced by their mean zero,
$$\hat s_{k+\tau}=\phi^\tau s_k . \tag{3}$$

The random walk is the limit φ = 1,
$$p_k=p_{k-1}+\xi_k , \tag{4}$$
in which every kick is kept forever. The value wanders without returning to any mean, and its spread grows with time. The kicks after day k have mean zero, so the best forecast of equation (1) is the last value for every horizon,
$$\hat p_{k+\tau}=p_k . \tag{5}$$ A process with this property, an expected future value equal to the present value, is called a martingale. Bachelier's model of prices is of this kind, and the efficient-market hypothesis is often stated as the hypothesis that prices are martingales.

<hr/>
## 3. Price changes and log returns

The next-day forecast of the index looks easy, because the value changes by less than one percent on most days. The forecast "tomorrow equals today", called persistence in the lecture, is therefore already within one percent of the truth on most days. This forecast tells us nothing about the market, however. Whoever owns shares earns or loses money through the change of the price, and the change is the part that persistence sets to zero. The quantity worth forecasting is the change.

The change in index points,
$$\Delta p_k=p_k-p_{k-1} , \tag{6}$$
depends on the level of the index. A change of 50 points was a large move in October 2016, when the index stood at about 2100 points, and a small one in October 2026, at about 7700 points. A comparison across the ten years needs the relative change, called the simple return,
$$R_k=\frac{p_k-p_{k-1}}{p_{k-1}} . \tag{7}$$
It is the fraction by which the price changed from one trading day to the next. A rise from 100 to 102 points is a return of 0.02, or 2%. The simple return also says what happens to money. An amount A invested in the index on day k − 1 is worth
$$A_k=A\,(1+R_k) \tag{8}$$
on day k, and after n days
$$A_n=A\,(1+R_1)(1+R_2)\cdots(1+R_n) . \tag{9}$$

Simple returns of consecutive days multiply, by equation (9), and this makes them awkward to combine. A gain of 10% followed by a loss of 10% does not bring the money back. From 100 we go to 110 and then to 99, a loss of 1%, although the two returns +0.10 and −0.10 add up to zero. The logarithm turns the product of equation (9) into a sum. The logarithmic return, or log return, of day k is
$$r_k=\ln(1+R_k)=\ln\frac{p_k}{p_{k-1}}=\ln p_k-\ln p_{k-1} , \tag{10}$$
the change of the logarithm of the price from one day to the next. The log returns of n consecutive days add up to the log return of the whole period,
$$r_1+r_2+\dots+r_n=\ln p_n-\ln p_0=\ln\frac{p_n}{p_0} , \tag{11}$$
since all intermediate prices cancel. For the gain and the loss of 10% the two log returns are $\ln 1.1=0.0953$ and $\ln 0.9=-0.1054$, and their sum, −0.0101, is the log return $\ln 0.99$ of the two days together. For the small daily changes of an index the two kinds of returns are nearly equal. The expansion of the logarithm gives
$$r_k=R_k-\tfrac12R_k^2+\dots , \tag{12}$$
so at a typical daily return of 1% the two differ by about 0.00005, one two-hundredth of the return itself.

Log returns have three properties that make them the standard variable of finance. They are relative, so a log return of 0.01 means a change of about 1% in 2016 and in 2026 alike. They add up over time, by equation (11), so the return of a week is the sum of five daily returns. And the record of log returns, unlike the price, fluctuates around a fixed mean close to zero, while the price wanders away from every level it has visited. A record whose statistical properties do not change in time is called stationary. All methods of the lecture assume a stationary signal, since they learn from the past a rule that they apply to the future.

A forecast of $r_{k+1}$ is a forecast of whether to buy or to sell. A positive forecast says that the price will rise, a negative one that it will fall, and the size says by how much. A forecaster with skill on returns could therefore make money, and the efficient-market hypothesis says that such skill should be very hard to find.

In [ ]:
logp = np.log(price)
ret = np.diff(logp)                       # log returns r_k = ln p_k - ln p_{k-1}, equation (10)
R_simple = np.diff(price) / price[:-1]    # simple returns, equation (7)
print(f"{len(ret)} daily log returns")
print(f"mean {ret.mean():.2e}, standard deviation {ret.std():.4f}")
print(f"largest rise {ret.max():+.4f} on {pd.Timestamp(dates[1:][ret.argmax()]).date()}, "
      f"largest fall {ret.min():+.4f} on {pd.Timestamp(dates[1:][ret.argmin()]).date()}")
print(f"sum of all log returns {ret.sum():.4f} = ln(p_last/p_first) = {np.log(price[-1] / price[0]):.4f}")
print(f"max |r_k - R_k| = {np.abs(ret - R_simple).max():.2e}, median |r_k - R_k| = {np.median(np.abs(ret - R_simple)):.2e}")
print(f"persistence on the price: median |p_(k+1) - p_k| / p_k = {np.median(np.abs(R_simple)):.4f}")

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(12, 9), layout="constrained", sharex=True)
axs[0].plot(dates, logp, color=UJBlue, lw=1.2); axs[0].set_ylabel(r"$\ln p_k$")
axs[1].plot(dates[1:], ret, color=UJNavy, lw=0.6); axs[1].set_ylabel(r"$r_k$")
axs[2].plot(dates[1:], np.abs(ret), color=DSOrange, lw=0.6); axs[2].set_ylabel(r"$|r_k|$")
axs[0].set_title("logarithm of the price, log returns and their size")
fig.autofmt_xdate()
plt.show()

The logarithm of the price in the upper panel wanders upwards. Its daily changes, the log returns in the middle panel, fluctuate around zero with a few very large spikes, in March 2020 and in April 2025, when new import tariffs of the United States were announced. The lower panel shows the size of the returns, $|r_k|$. Calm periods with small returns alternate with turbulent periods with large returns of both signs. Large changes follow large changes, an effect known as volatility clustering.

Daily returns of stock indices share a few statistical properties, called stylised facts, which have been found in many markets and periods [R. Cont, Quant. Finance **1**, 223 (2001), [doi:10.1080/713665670](https://doi.org/10.1080/713665670)]. The autocorrelation of the returns is close to zero already at a lag of one day, so a linear forecast from past returns has little to work with. The distribution of the returns has heavy tails, which means that very large returns are far more frequent than for a Gaussian distribution of the same variance, as Mandelbrot noticed for cotton prices [B. Mandelbrot, J. Business **36**, 394 (1963), [doi:10.1086/294632](https://doi.org/10.1086/294632)]. And the size of the returns, $|r_k|$ or $r_k^2$, has an autocorrelation that stays positive for weeks, which is the volatility clustering seen in the record. Volatility is the standard deviation of the returns over a period, the typical size of a daily move. Models in which today's variance depends on yesterday's squared returns, the ARCH and GARCH models, describe this effect [R. F. Engle, Econometrica **50**, 987 (1982), [doi:10.2307/1912773](https://doi.org/10.2307/1912773); T. Bollerslev, J. Econometrics **31**, 307 (1986), [doi:10.1016/0304-4076(86)90063-1](https://doi.org/10.1016/0304-4076(86)90063-1)].

The heaviness of the tails is measured by the excess kurtosis,
$$\kappa=\frac{\langle (r-\bar r)^4\rangle}{\langle (r-\bar r)^2\rangle^2}-3 , \tag{13}$$
where the angle brackets denote the average over the record and $\bar r$ the mean return. The fourth power weights the large values strongly, and the subtraction of 3 makes κ = 0 for a Gaussian distribution. The autocorrelation function at lag j,
$$\rho(j)=\frac{\langle (s_k-\bar s)(s_{k+j}-\bar s)\rangle}{\langle (s_k-\bar s)^2\rangle} , \tag{14}$$
is the correlation of the signal with itself shifted by j steps. It is 1 at j = 0 and lies between −1 and 1. For a record of n independent values, ρ(j) at j > 0 fluctuates around zero with a standard deviation of about $1/\sqrt n$, so values inside $\pm 2/\sqrt n$ are compatible with no correlation at all.

In [ ]:
z = (ret - ret.mean()) / ret.std()
kurt = scipy.stats.kurtosis(ret)            # excess kurtosis, equation (13)
n_beyond = int(np.sum(np.abs(z) > 4))
n_gauss = len(ret) * 2 * scipy.stats.norm.sf(4)
print(f"excess kurtosis {kurt:.2f} (Gaussian 0)")
print(f"returns beyond 4 standard deviations: {n_beyond} measured, {n_gauss:.2f} expected for a Gaussian")

J = 30
rho_r, rho_abs = acf(ret, nlags=J, fft=True), acf(np.abs(ret), nlags=J, fft=True)
band = 2 / np.sqrt(len(ret))
print(f"autocorrelation at lags 1, 2, 5, 20: returns {np.round(rho_r[[1, 2, 5, 20]], 3)}, "
      f"|returns| {np.round(rho_abs[[1, 2, 5, 20]], 3)}, band +-{band:.3f}")

fig, axs = plt.subplots(1, 2, figsize=(13, 4.6), layout="constrained")
bins = np.linspace(-8, 8, 81)
axs[0].hist(z, bins=bins, density=True, color=UJBlue, alpha=0.7, label="S&P 500")
axs[0].plot(bins, scipy.stats.norm.pdf(bins), color=DSOrange, lw=2, label="Gaussian")
axs[0].set_yscale("log"); axs[0].set_ylim(1e-4, 1)
axs[0].set_xlabel(r"$(r_k-\bar r)/\mathrm{std}$"); axs[0].set_ylabel("probability density"); axs[0].legend()
lags = np.arange(1, J + 1)
axs[1].bar(lags - 0.2, rho_r[1:], width=0.4, color=UJNavy, label=r"returns $r_k$")
axs[1].bar(lags + 0.2, rho_abs[1:], width=0.4, color=DSOrange, label=r"size $|r_k|$")
axs[1].axhspan(-band, band, color="0.85", zorder=0, label=r"$\pm 2/\sqrt{n}$")
axs[1].set_xlabel("lag $j$ [trading days]"); axs[1].set_ylabel(r"$\rho(j)$"); axs[1].legend()
plt.show()

The left panel compares the histogram of the standardised returns with the Gaussian density on a logarithmic scale, on which the tails become visible. The right panel shows the autocorrelation of the returns and of their size against the lag, with the grey band of values compatible with no correlation.

The power spectrum shows which periods are present in a record. It is estimated by the periodogram, the squared magnitude of the discrete Fourier transform of the record, and smoothed by the Welch method, which averages the periodograms of overlapping segments. Both estimates are also used in the analysis of central England weather. A record without memory, white noise, has a flat spectrum, the same power at every frequency. A record with memory concentrates its power at low frequencies, the slow changes.

In [ ]:
import scipy.signal
fig, ax = plt.subplots(figsize=(11, 4.6), layout="constrained")
for x, name, col in ((ret, r"returns $r_k$", UJNavy), (np.abs(ret), r"size $|r_k|$", DSOrange)):
    xs = (x - x.mean()) / x.std()
    f_w, P_w = scipy.signal.welch(xs, fs=252, nperseg=512, scaling="density")
    ax.loglog(f_w[1:], P_w[1:], color=col, lw=2, label=name)
    lo, hi = P_w[(f_w > 0) & (f_w < 5)].mean(), P_w[f_w > 50].mean()
    print(f"{name:16s} mean power below 5 cycles per year {lo:.4f}, above 50 cycles per year {hi:.4f}, ratio {lo / hi:.2f}")
ax.set_xlabel("frequency [cycles per year of 252 trading days]"); ax.set_ylabel("power spectral density (standardised)")
ax.legend()
plt.show()

<hr/>
## 4. Forecasting protocol and scores

For daily data, every signal of length N is split in time order into a fitting block, the first 60% of the samples, a validation block, the next 20%, and a test block, the last 20%. A model is fitted on the fitting block, its settings, such as the order of an AR model or the ridge strength, are chosen by the score on the validation block, the chosen model is fitted again on the fitting and validation blocks together, and it is scored once on the test block. Every signal is standardised with the mean and the standard deviation of its fitting block, so that all signals have values of order one.

A forecast is made from every origin k of a block. The model receives the values up to $s_k$ and returns the forecasts $\hat s_{k+\tau}$ for the horizons τ = 1, …, 20, in trading days for the S&P 500 data, so the longest horizon is about one month. All targets $s_{k+\tau}$ of the origins of a block lie inside that block, so no value of the test block is used in fitting. Models that need a window of past values receive the last L ≤ 20 values.

The score at horizon τ is the forecast capacity, the squared Pearson correlation between the forecasts and the targets over all origins of the test block,
$$C(\tau)=r(\tau)^2,\qquad r(\tau)=\frac{\sum_k\big(\hat s_{k+\tau}-\overline{\hat s}_\tau\big)\big(s_{k+\tau}-\overline s_\tau\big)}{\sqrt{\sum_k\big(\hat s_{k+\tau}-\overline{\hat s}_\tau\big)^2\,\sum_k\big(s_{k+\tau}-\overline s_\tau\big)^2}} , \tag{15}$$
where the bars are averages over the origins at fixed τ. C(τ) is the fraction of the variance of the targets that a straight-line fit on the forecasts explains. It is 1 when the forecasts lie on a straight line against the targets, whatever the slope and the offset of the line, and 0 when they are uncorrelated. The total capacity sums it over the horizons,
$$C_{\rm tot}=\sum_{\tau=1}^{20}C(\tau) , \tag{16}$$
with the maximum 20. The capacities for these horizons are on a smaller scale than those summed up to τ = 50 in the lecture.

A forecast without any skill still gives a small positive C(τ), since the correlation of n unrelated numbers fluctuates around zero with a standard deviation of about $1/\sqrt n$, and its square has the mean 1/n. With about 480 test origins this is a floor of about 0.002 for every C(τ) and about 0.04 for $C_{\rm tot}$. Capacities of this size mean no skill.

In [ ]:
TAU_MAX, L_MAX = 20, 20
TAUS = np.arange(1, TAU_MAX + 1)
N = len(ret)                                       # every signal of the exercise has this length
N_FIT, N_VAL = int(0.6 * N), int(0.2 * N)
K0 = L_MAX - 1                                     # first origin with a full window of L_MAX values
O_FIT = np.arange(K0, N_FIT - TAU_MAX)             # origins whose targets lie in the fitting block
O_VAL = np.arange(N_FIT, N_FIT + N_VAL - TAU_MAX)  # ... in the validation block
O_TRAIN = np.arange(K0, N_FIT + N_VAL - TAU_MAX)   # ... in fitting and validation blocks (final fit)
O_TEST = np.arange(N_FIT + N_VAL, N - TAU_MAX)     # ... in the test block
print(f"N = {N}: fitting block 0..{N_FIT - 1}, validation {N_FIT}..{N_FIT + N_VAL - 1}, test {N_FIT + N_VAL}..{N - 1}")
print(f"origins: fit {len(O_FIT)}, validation {len(O_VAL)}, train {len(O_TRAIN)}, test {len(O_TEST)}")
print(f"test block of the S&P 500 returns: {pd.Timestamp(dates[1:][N_FIT + N_VAL]).date()} to {pd.Timestamp(dates[-1]).date()}")
print(f"no-skill floor: 1/n = {1 / len(O_TEST):.4f} per horizon, {TAU_MAX / len(O_TEST):.3f} for C_tot")

def standardise(x):
    # mean and standard deviation of the fitting block only
    return (x - x[:N_FIT].mean()) / x[:N_FIT].std()

def windows(s, origins, L):
    # rows (s_k, s_(k-1), ..., s_(k-L+1)), newest value first; shape (len(origins), L)
    return s[np.asarray(origins)[:, None] - np.arange(L)[None, :]]

def targets(s, origins):
    # Y[i, tau-1] = s_(k_i + tau); shape (len(origins), TAU_MAX)
    return s[np.asarray(origins)[:, None] + TAUS[None, :]]

def capacity(P, Y):
    # C(tau) of equation (15) for every column; a constant forecast has r = 0
    Pc, Yc = P - P.mean(0), Y - Y.mean(0)
    den = np.sqrt((Pc ** 2).sum(0) * (Yc ** 2).sum(0))
    r = np.where(den > 0, (Pc * Yc).sum(0) / np.where(den > 0, den, 1.0), 0.0)
    return r ** 2

# check of capacity against scipy on random numbers
_P, _Y = np.random.default_rng(0).standard_normal((2, 300, 3))
assert np.allclose(capacity(_P, _Y), [scipy.stats.pearsonr(_P[:, j], _Y[:, j]).statistic ** 2 for j in range(3)])

We divide the record in time order into fitting, validation and test blocks. At an origin k, the model receives a window of the last L values and forecasts the value τ steps later. The input window is marked in blue and the target in orange. We repeat the forecast at every origin of a block and compare the predictions with the measured values.

In [ ]:
# sketch of the forecasting task: blocks, window of past values, forecast at the horizon tau
rng = np.random.default_rng(1)
t = np.arange(120)
x = np.sin(2 * np.pi * t / 37) + 0.3 * rng.standard_normal(120)
k, L_demo, tau_demo = 108, 12, 6
fig, ax = plt.subplots(figsize=(13, 4.2), layout="constrained")
for (a, b), name, col in (((0, 72), "fitting block", "#DCE6F2"), ((72, 96), "validation block", "#E8F1E4"),
                          ((96, 120), "test block", "#FBE7DA")):
    ax.axvspan(a - 0.5, b - 0.5, color=col, zorder=0)
    ax.text((a + b) / 2, 2.25, name, ha="center", fontsize=13)
ax.plot(t, x, "o-", color="0.6", ms=3, lw=1)
win = np.arange(k - L_demo + 1, k + 1)
ax.plot(win, x[win], "o", color=UJBlue, ms=7, label=f"window of the last L = {L_demo} values")
ax.plot(k + tau_demo, x[k + tau_demo], "*", color=DSOrange, ms=18, label=rf"target $s_{{k+\tau}}$, $\tau$ = {tau_demo}")
ax.annotate("", xy=(k + tau_demo, 1.6), xytext=(k, 1.6), arrowprops=dict(arrowstyle="->", lw=2, color=DSOrange))
ax.text(k + tau_demo / 2, 1.72, r"horizon $\tau$", ha="center", color=DSOrange, fontsize=13)
ax.axvline(k, color=UJNavy, ls=":", lw=1.5); ax.text(k, -2.3, "origin $k$", ha="center", color=UJNavy, fontsize=13)
ax.set_ylim(-3.2, 2.6); ax.set_xlabel("time step"); ax.set_ylabel("$s_k$"); ax.legend(loc="lower left", ncol=2)
plt.show()

For each signal, `plot_scatter` draws the forecasts of every model against the targets at the horizons τ = 1, 5 and 20, one row per model, with the diagonal $\hat s=s$ and the value of C(τ) in each panel. `plot_capacity` draws C(τ) of all models against τ, and `ctot_table` prints the total capacities.

In [ ]:
MODEL_COLORS = {"persistence": UJGray, "AR": UJBlue, "MA": "#7FB3E0", "ARMA": UJNavy, "NVAR": "#2A9D8F",
                "MLP": "#E9C46A", "RNN": "#B5651D", "LSTM": DSOrange}

def plot_scatter(name, preds, s, taus=(1, 5, 20)):
    Y = targets(s, O_TEST)
    models = list(preds)
    fig, axs = plt.subplots(len(models), len(taus), figsize=(3.4 * len(taus), 3.0 * len(models)),
                            layout="constrained", squeeze=False)
    lim = np.quantile(np.abs(Y), 0.995) * 1.1
    for i, m in enumerate(models):
        C = capacity(preds[m], Y)
        for j, t in enumerate(taus):
            ax = axs[i, j]
            ax.plot(Y[:, t - 1], preds[m][:, t - 1], ".", ms=3, alpha=0.5, color=MODEL_COLORS.get(m, UJBlue))
            ax.plot([-lim, lim], [-lim, lim], color="0.5", lw=0.8, ls="--")
            ax.set_xlim(-lim, lim)
            ax.text(0.04, 0.96, f"$C={C[t - 1]:.3f}$", transform=ax.transAxes, va="top", fontsize=12)
            if i == 0: ax.set_title(rf"$\tau={t}$")
            if i == len(models) - 1: ax.set_xlabel(r"true $s_{k+\tau}$")
            if j == 0: ax.set_ylabel(f"{m}\n" + r"forecast $\hat s_{k+\tau}$")
    fig.suptitle(name)
    plt.show()

def plot_capacity(results, names, theory=None, ylim=None):
    fig, axs = plt.subplots(1, len(names), figsize=(6.2 * len(names), 4.6), layout="constrained", squeeze=False)
    for ax, name in zip(axs[0], names):
        for m, C in results[name].items():
            ax.plot(TAUS, C, "o-", ms=4, color=MODEL_COLORS.get(m, UJBlue), label=m)
        if theory is not None and name in theory:
            ax.plot(TAUS, theory[name], "k--", lw=1.5, label="theory")
        ax.axhline(1 / len(O_TEST), color="0.6", lw=1, ls=":")
        ax.set_title(name); ax.set_xlabel(r"horizon $\tau$")
        if ylim: ax.set_ylim(*ylim)
    axs[0, 0].set_ylabel(r"$C(\tau)$"); axs[0, -1].legend(fontsize=11, loc="upper right")
    plt.show()

def ctot_table(results):
    tab = pd.DataFrame({name: {m: C.sum() for m, C in res.items()} for name, res in results.items()})
    print("total capacity C_tot = sum of C(tau) over tau = 1..20 (maximum 20)")
    print(tab.round(3).to_string(na_rep="-"))
    return tab

<hr/>
## 5. Models

Every model is a function that receives a standardised signal s and returns the forecasts for all test origins and all horizons, an array of shape (number of test origins, 20). Persistence, which forecasts every horizon by the last value $s_k$, is given as an example. The other seven functions are yours to write. The notebooks on the weather in central England and on the sunspot number apply the same seven models with the same settings to their records and contain complete implementations to start from, and the lecture notebooks explain every model in detail.

1. **AR**, the direct autoregressive model. For every horizon τ a linear readout of the window $(s_k,\dots,s_{k-L+1})$ plus an offset, fitted by ridge regression, all horizons at once as the columns of one target matrix. Choose L from {1, 2, 3, 5, 10, 20} and the ridge strength λ from {10⁻⁶, 10⁻², 1, 100} by $C_{\rm tot}$ on the validation block.
2. **MA**, the moving-average model MA(q), fitted by maximum likelihood with `statsmodels` (`ARIMA` with order (0, 0, q) and a constant). The τ-step forecast iterates the one-step model with the future errors set to zero. The past errors $e_k$ are the residuals of the fitted model on the whole record (`res.apply(s).resid`). Choose q from {1, 2, 3, 5} on the validation block.
3. **ARMA**, the model ARMA(p, q), fitted and iterated in the same way, with (p, q) from {(1, 1), (2, 1), (1, 2), (2, 2)}.
4. **NVAR**, the window of L values with stride κ, every κ-th value, together with all products of two of its entries, and a ridge readout per horizon. Choose L from {2, 3, 5, 8}, κ from {1, 2, 4} and λ as for AR.
5. **MLP**, `sklearn.neural_network.MLPRegressor` with tanh units on the window of the last 20 values, with 20 outputs, one per horizon. Choose one hidden layer of 32 or 64 units and the weight penalty `alpha` from {10⁻³, 10⁻¹, 1} on the validation block, with `early_stopping=True` and `random_state=0`.
6. **RNN**, an Elman recurrent network with 32 hidden units, run over the window of the last 20 values from the oldest to the newest, with a linear readout of the last hidden state to the 20 horizons. Train it with Adam on the mean squared error of the fitting block, with gradient clipping and early stopping on the validation loss.
7. **LSTM**, the same with an LSTM cell.

The direct AR model and NVAR both fit their readouts by ridge regression. For a feature matrix Φ with one row per origin and a target matrix Y with one column per horizon, the ridge loss is
$$\mathcal L(W,\mathbf b)=\big\|Y-\Phi W-\mathbf 1\mathbf b^\top\big\|^2+\lambda\|W\|^2 , \tag{17}$$
where $\mathbf 1$ is a column of ones and $\mathbf b$ the offsets of the horizons. Its minimum follows from the centred matrices $\Phi_c$ and $Y_c$, from which the column means are subtracted,
$$W=(\Phi_c^\top\Phi_c+\lambda I)^{-1}\Phi_c^\top Y_c,\qquad \mathbf b^\top=\overline Y-\overline\Phi\,W , \tag{18}$$
where the bars are the column means, so the offsets are not penalised.

The MA and ARMA models are fitted by maximum likelihood, since their past errors depend on the parameters themselves. With the statsmodels convention, the fitted ARMA(p, q) model reads
$$s_k-\mu=\sum_{i=1}^p\phi_i\,(s_{k-i}-\mu)+e_k+\sum_{j=1}^q\theta_j\,e_{k-j} , \tag{19}$$
where μ is the mean of the process and $e_k$ the one-step error. The τ-step forecast applies equation (19) repeatedly from the origin k, with the forecasts in place of the unknown future values and zero in place of the unknown future errors. An MA(q) forecast therefore returns to μ after q steps.

The functions receive only the signal. Use the origins `O_FIT`, `O_VAL`, `O_TRAIN` and `O_TEST` and the functions `windows`, `targets` and `capacity` of Section 4.

In [ ]:
CHOSEN = {}          # settings chosen on the validation block, filled in by the forecasters

def forecast_persistence(s):
    # s_(k+tau) is forecast by s_k for every horizon
    return np.repeat(s[O_TEST][:, None], TAU_MAX, axis=1)

In [ ]:
def forecast_ar(s):
    """Direct AR: ridge readout of the window, one column per horizon; L and lambda on the validation block.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

def forecast_ma(s):
    """MA(q) fitted with statsmodels, iterated with future errors set to zero; q on the validation block.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

def forecast_arma(s):
    """ARMA(p, q) fitted with statsmodels, iterated with future errors set to zero; (p, q) on the validation block.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

def forecast_nvar(s):
    """NVAR: window with stride, all quadratic products, ridge readout; L, stride and lambda on the validation block.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

def forecast_mlp(s):
    """MLPRegressor on the window of L_MAX values, 20 outputs; width and alpha on the validation block.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

def forecast_rnn(s):
    """Elman RNN with 32 units over the window, readout of the last hidden state; early stopping on the validation loss.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

def forecast_lstm(s):
    """LSTM with 32 units over the window, readout of the last hidden state; early stopping on the validation loss.
    Returns an array of shape (len(O_TEST), TAU_MAX)."""
    raise NotImplementedError

In [ ]:
FORECASTERS = {"persistence": forecast_persistence, "AR": forecast_ar, "MA": forecast_ma, "ARMA": forecast_arma,
               "NVAR": forecast_nvar, "MLP": forecast_mlp, "RNN": forecast_rnn, "LSTM": forecast_lstm}
RESULTS, PREDS = {}, {}

def run_models(name, s, models=tuple(FORECASTERS)):
    # runs every implemented model on the standardised signal s; unimplemented ones are skipped
    RESULTS[name], PREDS[name] = {}, {}
    Y = targets(s, O_TEST)
    for m in models:
        t0 = time.time()
        try:
            P = FORECASTERS[m](s)
        except NotImplementedError:
            continue
        PREDS[name][m], RESULTS[name][m] = P, capacity(P, Y)
        info = CHOSEN.get(m, "")
        print(f"{name:14s} {m:12s} C_tot {RESULTS[name][m].sum():7.3f}   C(1) {RESULTS[name][m][0]:.3f}   "
              f"{info}  ({time.time() - t0:.1f} s)")

<hr/>
## 6. Calibration on an AR(1) process

A forecasting procedure that reports a capacity should first be tested on a signal whose capacity is known. The AR(1) process of equation (2) is such a signal. Its best forecast is equation (3), and its capacity follows from the correlation of the forecast with the target. Multiplying equation (2), written for $s_{k+\tau}$, by $s_k$ and averaging, and repeating this τ times, gives the autocorrelation
$$\rho(\tau)=\phi^\tau , \tag{20}$$
since the kicks after time k are uncorrelated with $s_k$. The forecast of equation (3) is proportional to $s_k$, so its correlation with the target is $\rho(\tau)$, and
$$C(\tau)=\phi^{2\tau},\qquad C_{\rm tot}=\sum_{\tau=1}^{20}\phi^{2\tau}=\frac{\phi^2\big(1-\phi^{40}\big)}{1-\phi^2} . \tag{21}$$
For φ = 0.8 this gives C(1) = 0.64, C(5) = 0.107 and $C_{\rm tot}$ = 1.778. No model can exceed these values on average, since equation (3) is the best possible forecast. The measured values scatter around them, because the test block is finite.

The process has the same length as the S&P 500 returns and is generated with φ = 0.8 and unit noise variance.

In [ ]:
def ar1_process(n, phi=0.8, seed=0, burn=500):
    rng = np.random.default_rng(seed)
    xi = rng.standard_normal(n + burn)
    x = np.zeros(n + burn)
    for k in range(1, n + burn):
        x[k] = phi * x[k - 1] + xi[k]            # equation (2)
    return x[burn:]                              # the first values, still remembering x_0 = 0, are dropped

PHI = 0.8
s_ar1 = standardise(ar1_process(N, PHI))
C_theory = PHI ** (2 * TAUS)                     # equation (21)
print(f"theory: C(1) = {C_theory[0]:.3f}, C(5) = {C_theory[4]:.3f}, C_tot = {C_theory.sum():.3f}")
print(f"measured autocorrelation at lags 1..3: {np.round(acf(s_ar1, nlags=3)[1:], 3)} (theory {PHI}, {PHI**2:.2f}, {PHI**3:.3f})")

### Exercise 1
1. Run all models on the AR(1) signal, plot the scatter of forecasts against targets, plot C(τ) of all models together with the theory of equation (21) and print the table of total capacities.
2. Compare every model with the theory. Explain why persistence, which ignores φ, reaches the theoretical capacity, using the scale invariance of C(τ).
3. Explain why a model with more parameters cannot beat the theory on average, and why some models still end slightly above it on this test block.

In [ ]:
run_models("AR(1)", s_ar1)

In [ ]:
plot_scatter("AR(1) process", PREDS["AR(1)"], s_ar1)
plot_capacity(RESULTS, ["AR(1)"], theory={"AR(1)": C_theory}, ylim=(0, 0.75))

<hr/>
## 7. Price level

A first attempt is to forecast the price itself. The standardised closing value $p_k$ of the S&P 500 is the signal, and persistence and the AR model are the forecasters.

### Exercise 2
1. Run persistence and AR on the standardised price, plot the scatter at τ = 1, 5, 20 and C(τ), and print the total capacities.
2. Compare $C_{\rm tot}$ with the values of the lecture. Then convert the persistence forecast of the price into a forecast of the return from day k to day k + τ, and state what this forecast says about buying or selling.
3. Explain the high capacity with the random walk of equation (4). The test block covers about two years in which the index rose strongly. Describe what the correlation between $p_k$ and $p_{k+\tau}$ over the test origins measures when the level drifts by much more than its daily change.

In [ ]:
s_price = standardise(price[1:])                 # price on the days of the returns, same length N
print(f"standardised price: fitting block {s_price[:N_FIT].min():.2f} to {s_price[:N_FIT].max():.2f}, "
      f"test block {s_price[N_FIT + N_VAL:].min():.2f} to {s_price[N_FIT + N_VAL:].max():.2f}")
run_models("price", s_price, ["persistence", "AR"])
plot_scatter("S&P 500 price", PREDS["price"], s_price)
plot_capacity(RESULTS, ["price"], ylim=(0, 1.02))

<hr/>
## 8. Log returns

The quantity worth forecasting is the log return of equation (10). Any skill on it would be a trading signal. The efficient-market hypothesis says that the skill should be close to zero, and the autocorrelation in Section 3 already showed little linear memory. A nonlinear model might still find something that the autocorrelation misses.

A capacity close to zero can be compared with two references. The first is the no-skill floor of Section 4, 1/n per horizon. The second is a control signal, the same returns shuffled in time. Shuffling keeps every single value and therefore the whole distribution, with its heavy tails, and it destroys every dependence on the past. A model that scores the same on the original and on the shuffled returns has found no temporal structure.

### Exercise 3
1. Run all models on the standardised log returns and on the shuffled returns. Plot the scatter at τ = 1, 5, 20, plot C(τ) for both signals and print the table of total capacities.
2. Compare $C_{\rm tot}$ of the returns with the no-skill floor, with the shuffled control and with the values of the lecture for Mackey–Glass and the Santa Fe laser.
3. Describe the scatter plots. Look at the range of the forecasts compared with the range of the targets, and explain what a model that has learned nothing should forecast.
4. Repeat the AR fit on 100 independent shufflings of the returns and compare $C_{\rm tot}$ of the original returns with the distribution obtained. This is a permutation test.
5. A trader cares about the sign. For τ = 1, compute for every model the fraction of test days on which the sign of the forecast equals the sign of the true return, and compare it with the forecast that always predicts a rise.

In [ ]:
s_ret = standardise(ret)
s_shuf = standardise(np.random.default_rng(0).permutation(ret))    # the same values in random order
run_models("returns", s_ret)
run_models("shuffled returns", s_shuf)

In [ ]:
plot_scatter("S&P 500 log returns", PREDS["returns"], s_ret)

In [ ]:
plot_capacity(RESULTS, ["returns", "shuffled returns"], ylim=(0, 0.05))
ctot_table({k: RESULTS[k] for k in ("returns", "shuffled returns")});

<hr/>
## 9. Size of the daily move

The returns showed no usable memory, but their size did. The autocorrelation of $|r_k|$ in Section 3 stays positive for weeks. A forecast of $|r_{k+\tau}|$ says how large the move will be without saying in which direction, which is the information needed to price insurance against large moves and to set the risk of a portfolio.

### Exercise 4
1. Run all models on the standardised $|r_k|$ and on the absolute values of the shuffled returns. Plot the scatter, C(τ) and the table of total capacities.
2. Compare with the returns of Exercise 3. Explain how the same record can be close to unpredictable in its sign and partly predictable in its size.
3. Relate the decay of C(τ) with τ to the autocorrelation of $|r_k|$ measured in Section 3, and name the models that gain over persistence.

In [ ]:
s_abs = standardise(np.abs(ret))
s_abs_shuf = standardise(np.abs(np.random.default_rng(0).permutation(ret)))
run_models("|returns|", s_abs)
run_models("shuffled |returns|", s_abs_shuf)

In [ ]:
plot_scatter("size of the S&P 500 log returns", PREDS["|returns|"], s_abs)
plot_capacity(RESULTS, ["|returns|", "shuffled |returns|"], ylim=(0, 0.15))
ctot_table({k: RESULTS[k] for k in ("|returns|", "shuffled |returns|")});

<hr/>
## 10. Delay embedding of the returns


The hidden state of a deterministic system can be reconstructed from one measured coordinate using the delay vector
$$\mathbf u_k=\big(s_k,\;s_{k-\kappa},\;\dots,\;s_{k-(m-1)\kappa}\big) , \tag{22}$$
built from m values of the record spaced by the delay κ. By the theorem of Takens, for a large enough embedding dimension m the cloud of delay vectors is a faithful picture of the attractor. Mutual information determines κ from the dependence between delayed values, and false nearest neighbours determine m from the geometry of the reconstructed record.

The delay κ is taken at the first local minimum of the mutual information between $s_k$ and $s_{k-\kappa}$ [A. M. Fraser, H. L. Swinney, Phys. Rev. A **33**, 1134 (1986), [doi:10.1103/PhysRevA.33.1134](https://doi.org/10.1103/PhysRevA.33.1134)]. The range of the record is divided into 16 bins of equal width. For a delay κ we count, over all pairs $(s_{k-\kappa},s_k)$, the fraction $p_{ij}(\kappa)$ of pairs whose first value falls into bin i and second value into bin j, and the fractions $p_i$ and $p_j$ of single values in each bin. The mutual information is
$$I(\kappa)=\sum_{i,j}p_{ij}(\kappa)\,\ln\frac{p_{ij}(\kappa)}{p_i\,p_j} , \tag{23}$$
measured in nats. It is zero when the two values are independent, since then $p_{ij}=p_ip_j$ and every logarithm vanishes, and it grows with the dependence of the two values, linear or nonlinear. The autocorrelation of equation (14) measures only the linear part of this dependence. At the first minimum of I(κ) a new delay coordinate adds the most new information while it still belongs to the same stretch of motion.

The embedding dimension m is found with false nearest neighbours [M. B. Kennel, R. Brown, H. D. I. Abarbanel, Phys. Rev. A **45**, 3403 (1992), [doi:10.1103/PhysRevA.45.3403](https://doi.org/10.1103/PhysRevA.45.3403)]. If m is too small, the picture of the attractor is folded onto itself and points from distant parts of the orbit appear as neighbours. For every delay vector $\mathbf u_i^{(m)}$ of dimension m we find its nearest neighbour $\mathbf u_j^{(m)}$, excluding the points with $|i-j|\le w$, which are close only because they are close in time. This exclusion is called the Theiler window. The pair is a false neighbour if the next delay coordinate separates the two points strongly,
$$\frac{\big|s_{i-m\kappa}-s_{j-m\kappa}\big|}{\big\|\mathbf u_i^{(m)}-\mathbf u_j^{(m)}\big\|}>R_{\rm tol}=15 . \tag{24}$$
The fraction of false neighbours is computed for m = 1, 2, …, and the embedding dimension is the smallest m at which it falls below 5%. The ratio test alone is unreliable at large m. The distance between nearest neighbours grows with m, because every added coordinate adds to it, and the ratio of equation (24) becomes small for any record, also for noise. Kennel and co-workers therefore add an absolute test. The pair is also counted as false if its distance in m + 1 dimensions exceeds twice the standard deviation $\sigma_s$ of the record,
$$\frac{\Big(\big\|\mathbf u_i^{(m)}-\mathbf u_j^{(m)}\big\|^2+\big|s_{i-m\kappa}-s_{j-m\kappa}\big|^2\Big)^{1/2}}{\sigma_s}>2 , \tag{25}$$
which identifies neighbours separated by a large distance relative to the variation in the record. We compute both fractions and choose the embedding dimension with the ratio test. The combined fraction, which counts a pair as false under the ratio test or under the absolute test, distinguishes deterministic from stochastic records. For a low-dimensional deterministic law it stays near zero once the attractor is unfolded. For a stochastic record new random influences enter every coordinate, the nearest neighbours stay far apart in the sense of equation (25), and the combined fraction stays high and grows with m.

The embedding window has the length $(m-1)\kappa+1$ samples and can serve directly as the input of the AR model and of NVAR, with L = m values taken with the stride κ.

### Exercise 5
1. Compute the mutual information of equation (23) of the log returns up to a delay of 20 days and look for its first minimum. Compute the fraction of false neighbours of equation (24) for m = 1, …, 8 with a Theiler window of 5 days. The notebook on the weather in central England shows the code.
2. Explain the result for a record without memory, and compare it with the sunspot record.

<hr/>
## 11. All signals and models

### Exercise 6
1. Print the table of total capacities of all signals and models.
2. Write a short report of one page. For each signal state whether it contains structure that the past fixes, which models find it, and how you know that a small capacity means no skill. Include the scatter plot and the C(τ) plot that you find most informative.
3. Name one measured signal from your own field of research and state, with reasons, which of the signals of this exercise it resembles most.

In [ ]:
table = ctot_table(RESULTS)
print(f"\ntotal runtime {time.time() - t_start:.0f} s")

## References

- L. Bachelier, *Théorie de la spéculation*, Ann. Sci. Éc. Norm. Supér. **17**, 21 (1900), [doi:10.24033/asens.476](https://doi.org/10.24033/asens.476)
- A. Einstein, *Über die von der molekularkinetischen Theorie der Wärme geforderte Bewegung von in ruhenden Flüssigkeiten suspendierten Teilchen*, Ann. Phys. **17**, 549 (1905), [doi:10.1002/andp.19053220806](https://doi.org/10.1002/andp.19053220806)
- E. F. Fama, *Efficient capital markets: a review of theory and empirical work*, J. Finance **25**, 383 (1970), [doi:10.1111/j.1540-6261.1970.tb00518.x](https://doi.org/10.1111/j.1540-6261.1970.tb00518.x)
- B. Mandelbrot, *The variation of certain speculative prices*, J. Business **36**, 394 (1963), [doi:10.1086/294632](https://doi.org/10.1086/294632)
- R. Cont, *Empirical properties of asset returns: stylized facts and statistical issues*, Quant. Finance **1**, 223 (2001), [doi:10.1080/713665670](https://doi.org/10.1080/713665670)
- R. F. Engle, *Autoregressive conditional heteroscedasticity with estimates of the variance of United Kingdom inflation*, Econometrica **50**, 987 (1982), [doi:10.2307/1912773](https://doi.org/10.2307/1912773)
- T. Bollerslev, *Generalized autoregressive conditional heteroskedasticity*, J. Econometrics **31**, 307 (1986), [doi:10.1016/0304-4076(86)90063-1](https://doi.org/10.1016/0304-4076(86)90063-1)
- S&P Dow Jones Indices LLC, *S&P 500* [SP500], retrieved from FRED, Federal Reserve Bank of St. Louis, <https://fred.stlouisfed.org/series/SP500>